# Cow Behavior Dataset Setup

Automated download and setup of the CBVD-5 Cow Behavior Video Dataset from Kaggle.

## Prerequisites

1. Kaggle account at https://www.kaggle.com
2. Kaggle API credentials (`kaggle.json`):
   - Profile → Account → Create New API Token
   - Place file at `~/.kaggle/kaggle.json` (Linux/Mac) or `C:\Users\<username>\.kaggle\kaggle.json` (Windows)

## Process

This notebook will:
- Install required packages
- Download CBVD-5 dataset (~6GB)
- Extract `videos/` and `labelframes/` directories
- Download YOLO pre-trained weights
- Verify setup completion

## Install Dependencies and Import Libraries

In [ ]:
import os
import zipfile
import shutil
import urllib.request
from pathlib import Path
from tqdm import tqdm

# Initialize Kaggle API
try:
    import kaggle
    from kaggle.api.kaggle_api_extended import KaggleApi
    
    api = KaggleApi()
    api.authenticate()
    user = api.get_config_value('username')
    print(f"Kaggle API authenticated as: {user}")
    
except Exception as e:
    print(f"Kaggle API error: {e}")
    print("Please ensure kaggle.json is in the correct location with proper credentials")
    raise

In [ ]:
## Setup Paths and Check Current Status

# Define paths
PROJECT_ROOT = Path.cwd()
DATA_DIR = PROJECT_ROOT / "data"
DOWNLOAD_DIR = PROJECT_ROOT / "temp_download"
VIDEOS_DIR = DATA_DIR / "videos"
LABELFRAMES_DIR = DATA_DIR / "labelframes"

# Create directories
DOWNLOAD_DIR.mkdir(exist_ok=True)
DATA_DIR.mkdir(exist_ok=True)

# Check current status
videos_exist = VIDEOS_DIR.exists() and any(VIDEOS_DIR.glob('*.mp4'))
labelframes_exist = LABELFRAMES_DIR.exists() and any(LABELFRAMES_DIR.rglob('*.jpg'))

print(f"Videos directory: {'EXISTS' if videos_exist else 'MISSING'}")
print(f"Labelframes directory: {'EXISTS' if labelframes_exist else 'MISSING'}")

if videos_exist and labelframes_exist:
    print("Dataset already exists - skipping download")
else:
    print("Dataset download required")

In [ ]:
## Download Dataset

DATASET_NAME = "fandaoerji/cbvd-5cow-behavior-video-dataset"
DATASET_ZIP = "cbvd-5cow-behavior-video-dataset.zip"

def download_dataset():
    print(f"Downloading {DATASET_NAME} (~6GB)...")
    
    api.dataset_download_files(
        dataset=DATASET_NAME,
        path=DOWNLOAD_DIR,
        quiet=False
    )
    
    zip_path = DOWNLOAD_DIR / DATASET_ZIP
    if zip_path.exists():
        size_gb = zip_path.stat().st_size / (1024**3)
        print(f"Download completed: {size_gb:.2f} GB")
        return zip_path
    else:
        raise FileNotFoundError(f"Downloaded file not found: {zip_path}")

# Download if needed
if not (videos_exist and labelframes_exist):
    zip_path = download_dataset()
else:
    zip_path = None

In [ ]:
## Extract Files

def extract_needed_directories(zip_path):
    print("Extracting dataset files...")
    
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        all_files = zip_ref.namelist()
        
        videos_files = [f for f in all_files if 'videos/' in f and f.endswith('.mp4')]
        labelframes_files = [f for f in all_files if 'labelframes/' in f and f.endswith('.jpg')]
        
        print(f"Found {len(videos_files)} video files")
        print(f"Found {len(labelframes_files)} labelframe images")
        
        # Extract with progress
        if videos_files and not videos_exist:
            for file in tqdm(videos_files, desc="Extracting videos"):
                zip_ref.extract(file, DOWNLOAD_DIR)
        
        if labelframes_files and not labelframes_exist:
            for file in tqdm(labelframes_files, desc="Extracting labelframes"):
                zip_ref.extract(file, DOWNLOAD_DIR)

if zip_path and not (videos_exist and labelframes_exist):
    extract_needed_directories(zip_path)

In [ ]:
## Organize Files

def move_to_project_structure():
    print("Organizing files into project structure...")
    
    # Find extracted directories
    extracted_videos = None
    extracted_labelframes = None
    
    for root, dirs, files in os.walk(DOWNLOAD_DIR):
        if 'videos' in root and any(f.endswith('.mp4') for f in files):
            extracted_videos = root
        if 'labelframes' in root and any(f.endswith('.jpg') for f in files):
            extracted_labelframes = root
    
    # Move directories
    if extracted_videos and not videos_exist:
        if VIDEOS_DIR.exists():
            shutil.rmtree(VIDEOS_DIR)
        shutil.move(extracted_videos, VIDEOS_DIR)
        print("Videos moved successfully")
    
    if extracted_labelframes and not labelframes_exist:
        if LABELFRAMES_DIR.exists():
            shutil.rmtree(LABELFRAMES_DIR)
        shutil.move(extracted_labelframes, LABELFRAMES_DIR)
        print("Labelframes moved successfully")

if zip_path and not (videos_exist and labelframes_exist):
    move_to_project_structure()

In [ ]:
## Cleanup

if DOWNLOAD_DIR.exists() and zip_path:
    print("Cleaning up temporary files...")
    try:
        shutil.rmtree(DOWNLOAD_DIR)
        print("Cleanup completed")
    except Exception as e:
        print(f"Warning: Could not clean up {DOWNLOAD_DIR}: {e}")

In [ ]:
## Verify Setup

# Verify final setup
print("Verifying dataset setup...")

# Check directories and files
video_count = len(list(VIDEOS_DIR.glob('*.mp4'))) if VIDEOS_DIR.exists() else 0
labelframe_count = len(list(LABELFRAMES_DIR.rglob('*.jpg'))) if LABELFRAMES_DIR.exists() else 0

# Results
checks = [
    ("Videos", video_count > 0, f"{video_count} files"),
    ("Labelframes", labelframe_count > 0, f"{labelframe_count} files"),
]

all_good = True
for name, passed, detail in checks:
    status = "OK" if passed else "MISSING"
    print(f"{name}: {status} ({detail})")
    if not passed:
        all_good = False

print("\n" + "="*40)
if all_good:
    print("SETUP COMPLETE")
    print("\nNext steps:")
    print("1. 01_bbox_crops.ipynb")
    print("2. 02_yolo_oneclass_from_via.ipynb")
    print("3. 05_vit_behavior_classifier.ipynb")
    print("4. 06_cow_detection_and_behavior_pipeline.ipynb")
else:
    print("SETUP INCOMPLETE - check missing items above")

In [ ]:
## Verify Setup

# Verify final setup
print("Verifying dataset setup...")

# Check directories and files
video_count = len(list(VIDEOS_DIR.glob('*.mp4'))) if VIDEOS_DIR.exists() else 0
labelframe_count = len(list(LABELFRAMES_DIR.rglob('*.jpg'))) if LABELFRAMES_DIR.exists() else 0
yolo8_exists = (PROJECT_ROOT / 'yolov8n.pt').exists()
yolo11_exists = (PROJECT_ROOT / 'yolo11n.pt').exists()

# Results
checks = [
    ("Videos", video_count > 0, f"{video_count} files"),
    ("Labelframes", labelframe_count > 0, f"{labelframe_count} files"),
    ("YOLOv8 weights", yolo8_exists, "yolov8n.pt"),
    ("YOLO11 weights", yolo11_exists, "yolo11n.pt"),
]

all_good = True
for name, passed, detail in checks:
    status = "OK" if passed else "MISSING"
    print(f"{name}: {status} ({detail})")
    if not passed:
        all_good = False

print("\n" + "="*40)
if all_good:
    print("SETUP COMPLETE")
    print("\nNext steps:")
    print("1. 01_bbox_crops.ipynb")
    print("2. 02_yolo_oneclass_from_via.ipynb")
    print("3. 05_vit_behavior_classifier.ipynb")
    print("4. 06_cow_detection_and_behavior_pipeline.ipynb")
else:
    print("SETUP INCOMPLETE - check missing items above")

In [ ]:
Dataset setup is now complete. YOLO weights will be downloaded automatically when running the training notebooks.

## 🎯 Next Steps

If the setup completed successfully, you're ready to run the main analysis notebooks!

### Recommended execution order:

1. **`01_bbox_crops.ipynb`** - Extract behavior crops from VIA annotations
2. **`02_yolo_oneclass_from_via.ipynb`** - Train YOLO cow detector  
3. **`05_vit_behavior_classifier.ipynb`** - Train ViT behavior classifier
4. **`06_cow_detection_and_behavior_pipeline.ipynb`** - End-to-end pipeline demo

### If you encounter issues:

- **Kaggle API errors**: Check your `kaggle.json` credentials file
- **Download failures**: Try running this notebook again
- **Disk space**: Ensure you have at least 7GB free space
- **Network issues**: The download is large (~6GB), ensure stable internet

---

**Happy analyzing! 🐄📊**